# Day 53 — LLM APIs: Streaming, Error Handling, Tool Calling & Multi-Provider
### Anthropic SDK + OpenAI SDK — 4 distinct use cases

## 1. Setup — Clients & Imports

In [1]:
import os
import json
import time
import anthropic

client = anthropic.Anthropic()
HAIKU = "claude-haiku-4-5"

print("Anthropic SDK version:", anthropic.__version__)
print("Client ready:", client is not None)

Anthropic SDK version: 0.111.0
Client ready: True


## 2. Use Case 1 — Streaming Responses

In [2]:
print("Streaming response — tokens arrive as generated:\n")

with client.messages.stream(
    model=HAIKU,
    max_tokens=300,
    messages=[{"role": "user", "content": "Explain what a transformer neural network is in 3 sentences."}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

print("\n\n--- stream complete ---")

Streaming response — tokens arrive as generated:

# Transformer Neural Network

A transformer is a deep learning architecture that processes input data (like text) by using "attention mechanisms" to weigh the importance of different parts of the input simultaneously, rather than processing it sequentially. This allows transformers to capture long-range relationships and dependencies efficiently, making them excellent for tasks like language translation and text generation. Transformers power modern large language models like GPT and BERT because they can be parallelized during training and scale effectively to handle massive amounts of data.

--- stream complete ---


## 3. Use Case 2 — Robust Error Handling

In [3]:
def ask_with_retry(prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model=HAIKU,
                max_tokens=200,
                messages=[{"role": "user", "content": prompt}]
            )
            return response.content[0].text

        except anthropic.RateLimitError:
            wait = 2 ** attempt
            print(f"Rate limit hit. Waiting {wait}s before retry {attempt+1}/{max_retries}...")
            time.sleep(wait)

        except anthropic.AuthenticationError:
            print("Authentication failed -- check your API key.")
            raise

        except anthropic.APIConnectionError:
            print(f"Connection error on attempt {attempt+1}. Retrying...")
            time.sleep(1)

        except anthropic.APIStatusError as e:
            print(f"API error {e.status_code}: {e.message}")
            raise

    raise RuntimeError(f"Failed after {max_retries} retries")

result = ask_with_retry("What is the capital of Japan?")
print("Response:", result)

Response: The capital of Japan is Tokyo. It's the largest city in Japan and has been the capital since 1868.


## 4. Use Case 3 — Full Tool Calling Loop

In [4]:
import random

def get_stock_price(symbol):
    prices = {"AAPL": 189.50, "GOOGL": 175.20, "MSFT": 415.80, "TSLA": 242.10}
    return prices.get(symbol.upper(), round(random.uniform(50, 500), 2))

tools = [
    {
        "name": "get_stock_price",
        "description": "Get the current stock price for a given ticker symbol",
        "input_schema": {
            "type": "object",
            "properties": {
                "symbol": {"type": "string", "description": "Stock ticker symbol e.g. AAPL"}
            },
            "required": ["symbol"]
        }
    }
]

messages = [{"role": "user", "content": "What is the current stock price of Apple and Microsoft?"}]

response = client.messages.create(model=HAIKU, max_tokens=500, tools=tools, messages=messages)
print("Stop reason:", response.stop_reason)
print("Content blocks:", [b.type for b in response.content])

Stop reason: tool_use
Content blocks: ['tool_use', 'tool_use']


In [5]:
tool_results = []
for block in response.content:
    if block.type == "tool_use":
        price = get_stock_price(block.input["symbol"])
        print(f"Called {block.name}({block.input['symbol']}) -> ${price}")
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": str(price)
        })

messages.append({"role": "assistant", "content": response.content})
messages.append({"role": "user", "content": tool_results})

final = client.messages.create(model=HAIKU, max_tokens=300, tools=tools, messages=messages)
print("\nFinal answer:")
print(final.content[0].text)

Called get_stock_price(AAPL) -> $189.5
Called get_stock_price(MSFT) -> $415.8

Final answer:
The current stock prices are:

- **Apple (AAPL)**: $189.50
- **Microsoft (MSFT)**: $415.80


## 5. Use Case 4 — OpenAI SDK Comparison

In [7]:
# OpenAI SDK — same task, different SDK structure
# (requires billing setup at platform.openai.com to run live)

# from openai import OpenAI
# openai_client = OpenAI()  # reads OPENAI_API_KEY from environment

# response = openai_client.chat.completions.create(
#     model="gpt-4o-mini",
#     max_tokens=200,
#     messages=[
#         {"role": "system", "content": "Classify sentiment in one word only."},
#         {"role": "user", "content": "The food was cold but the service was excellent."}
#     ]
# )
# print(response.choices[0].message.content)   # -> neutral
# print(response.usage.total_tokens)

# --- SDK COMPARISON ---
print("Anthropic SDK                    | OpenAI SDK")
print("-" * 60)
print("anthropic.Anthropic()            | OpenAI()")
print("client.messages.create(...)      | client.chat.completions.create(...)")
print("model='claude-haiku-4-5'         | model='gpt-4o-mini'")
print("messages=[{role, content}]       | messages=[{role, content}]  <- identical")
print("response.content[0].text         | response.choices[0].message.content")
print("response.usage.input_tokens      | response.usage.prompt_tokens")
print("system= parameter (top-level)    | {role:'system'} in messages list")

Anthropic SDK                    | OpenAI SDK
------------------------------------------------------------
anthropic.Anthropic()            | OpenAI()
client.messages.create(...)      | client.chat.completions.create(...)
model='claude-haiku-4-5'         | model='gpt-4o-mini'
messages=[{role, content}]       | messages=[{role, content}]  <- identical
response.content[0].text         | response.choices[0].message.content
response.usage.input_tokens      | response.usage.prompt_tokens
system= parameter (top-level)    | {role:'system'} in messages list


## 6. Summary — What We Built Today

In [8]:
summary = {
    "use_case_1": "Streaming — tokens arrive progressively via client.messages.stream()",
    "use_case_2": "Error handling — retry logic for RateLimitError, AuthenticationError, ConnectionError",
    "use_case_3": "Full tool loop — model requests tools, we execute, feed results back, get final answer",
    "use_case_4": "SDK comparison — Anthropic vs OpenAI structural differences side by side",
    "key_insight": "Both SDKs follow same JSON-in/JSON-out pattern — switching providers is a 2-line change"
}

for k, v in summary.items():
    print(f"{k}: {v}")

use_case_1: Streaming — tokens arrive progressively via client.messages.stream()
use_case_2: Error handling — retry logic for RateLimitError, AuthenticationError, ConnectionError
use_case_3: Full tool loop — model requests tools, we execute, feed results back, get final answer
use_case_4: SDK comparison — Anthropic vs OpenAI structural differences side by side
key_insight: Both SDKs follow same JSON-in/JSON-out pattern — switching providers is a 2-line change
